## biLSTM + Attention 모델 설계 시도

#### 참고 
* https://github.com/ukairia777/tensorflow-nlp-tutorial/blob/main/14.%20Seq2Seq%20(NMT)/14-2.%20word_level_seq2seq.ipynb

In [16]:
import tensorflow as tf
tf.__version__

'2.4.1'

In [17]:
import re
import os
import unicodedata
import urllib3
import zipfile
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Embedding, Bidirectional, LSTM, Concatenate, Dropout
from tensorflow.keras import Input, Model
from tensorflow.keras import optimizers

In [18]:
num_samples = 33000

In [19]:
path = os.getcwd()
file_folder = 'fra-eng'
file_name = 'fra.txt'
data_path = os.path.join(path, file_folder)
file_path = os.path.join(data_path, file_name)

In [20]:
file_path

'C:\\Users\\PC\\Desktop\\code\\fra-eng\\fra.txt'

In [21]:
def unicode_to_ascii(s):
  # 프랑스어 악센트(accent) 삭제
  # 예시 : 'déjà diné' -> deja dine
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

In [22]:
def preprocess_sentence(sent):
  # 악센트 삭제 함수 호출
    sent = unicode_to_ascii(sent.lower())

  # 단어와 구두점 사이에 공백을 만듭니다.
  # Ex) "he is a boy." => "he is a boy ."
    sent = re.sub(r"([?.!,¿])", r" \1", sent)

  # (a-z, A-Z, ".", "?", "!", ",") 이들을 제외하고는 전부 공백으로 변환합니다.
    sent = re.sub(r"[^a-zA-Z!.?]+", r" ", sent)

  # 다수 개의 공백을 하나의 공백으로 치환
    sent = re.sub(r"\s+", " ", sent)
    return sent

In [23]:
# 전처리 테스트
en_sent = u"Have you had dinner?"
fr_sent = u"Avez-vous déjà diné?"

print('전처리 전 영어 문장 :', en_sent)
print('전처리 후 영어 문장 :',preprocess_sentence(en_sent))
print('전처리 전 프랑스어 문장 :', fr_sent)
print('전처리 후 프랑스어 문장 :', preprocess_sentence(fr_sent))

전처리 전 영어 문장 : Have you had dinner?
전처리 후 영어 문장 : have you had dinner ?
전처리 전 프랑스어 문장 : Avez-vous déjà diné?
전처리 후 프랑스어 문장 : avez vous deja dine ?


In [24]:
def load_preprocessed_data():
    encoder_input, decoder_input, decoder_target = [], [], []

    with open(file_path, "r", encoding = 'UTF8') as lines:
        for i, line in enumerate(lines):
      # source 데이터와 target 데이터 분리
            src_line, tar_line, _ = line.strip().split('\t')

      # source 데이터 전처리
            src_line = [w for w in preprocess_sentence(src_line).split()]

      # target 데이터 전처리
            tar_line = preprocess_sentence(tar_line)
            tar_line_in = [w for w in (" " + tar_line).split()]
            tar_line_out = [w for w in (tar_line + " ").split()]

            encoder_input.append(src_line)
            decoder_input.append(tar_line_in)
            decoder_target.append(tar_line_out)

            if i == num_samples - 1:
                break

    return encoder_input, decoder_input, decoder_target

In [25]:
sents_en_in, sents_fra_in, sents_fra_out = load_preprocessed_data()

In [26]:
print('인코더의 입력 :',sents_en_in[:5])
print('디코더의 입력 :',sents_fra_in[:5])
print('디코더의 레이블 :',sents_fra_out[:5])

인코더의 입력 : [['go', '.'], ['go', '.'], ['go', '.'], ['go', '.'], ['hi', '.']]
디코더의 입력 : [['va', '!'], ['marche', '.'], ['en', 'route', '!'], ['bouge', '!'], ['salut', '!']]
디코더의 레이블 : [['va', '!'], ['marche', '.'], ['en', 'route', '!'], ['bouge', '!'], ['salut', '!']]


In [27]:
tokenizer_en = Tokenizer(filters = "", lower = False)
tokenizer_en.fit_on_texts(sents_en_in)
encoder_input = tokenizer_en.texts_to_sequences(sents_en_in)
encoder_input = pad_sequences(encoder_input, padding="post")

tokenizer_fra = Tokenizer(filters="", lower=False)
tokenizer_fra.fit_on_texts(sents_fra_in)
tokenizer_fra.fit_on_texts(sents_fra_out)

decoder_input = tokenizer_fra.texts_to_sequences(sents_fra_in)
decoder_input = pad_sequences(decoder_input, padding="post")

decoder_target = tokenizer_fra.texts_to_sequences(sents_fra_out)
decoder_target = pad_sequences(decoder_target, padding="post")

In [28]:
print('인코더의 입력의 크기(shape) :',encoder_input.shape)
print('디코더의 입력의 크기(shape) :',decoder_input.shape)
print('디코더의 레이블의 크기(shape) :',decoder_target.shape)

인코더의 입력의 크기(shape) : (33000, 8)
디코더의 입력의 크기(shape) : (33000, 15)
디코더의 레이블의 크기(shape) : (33000, 15)


In [29]:
src_to_index = tokenizer_en.word_index
index_to_src = tokenizer_en.index_word
tar_to_index = tokenizer_fra.word_index
index_to_tar = tokenizer_fra.index_word

In [30]:
max_src_len = encoder_input.shape[1]
max_tar_len = decoder_input.shape[1]
print('source 문장의 최대 길이 :',max_src_len)
print('target 문장의 최대 길이 :',max_tar_len)

source 문장의 최대 길이 : 8
target 문장의 최대 길이 : 15


In [31]:
src_vocab_size = len(tokenizer_en.word_index) + 1
tar_vocab_size = len(tokenizer_fra.word_index) + 1
print("영어 단어 집합의 크기 : {:d}, 프랑스어 단어 집합의 크기 : {:d}".format(src_vocab_size, tar_vocab_size))

영어 단어 집합의 크기 : 4672, 프랑스어 단어 집합의 크기 : 8135


In [32]:
indices = np.arange(encoder_input.shape[0])
np.random.shuffle(indices)
print('랜덤 시퀀스 :',indices)

랜덤 시퀀스 : [  358  2027  3377 ...  8741 16064 17466]


In [33]:
encoder_input = encoder_input[indices]
decoder_input = decoder_input[indices]
decoder_target = decoder_target[indices]

In [34]:
n_of_val = int(33000*0.1)
print('검증 데이터의 개수 :',n_of_val)

검증 데이터의 개수 : 3300


In [35]:
encoder_input_train = encoder_input[:-n_of_val]
decoder_input_train = decoder_input[:-n_of_val]
decoder_target_train = decoder_target[:-n_of_val]

encoder_input_test = encoder_input[-n_of_val:]
decoder_input_test = decoder_input[-n_of_val:]
decoder_target_test = decoder_target[-n_of_val:]

In [36]:
print('훈련 source 데이터의 크기 :',encoder_input_train.shape)
print('훈련 target 데이터의 크기 :',decoder_input_train.shape)
print('훈련 target 레이블의 크기 :',decoder_target_train.shape)
print('테스트 source 데이터의 크기 :',encoder_input_test.shape)
print('테스트 target 데이터의 크기 :',decoder_input_test.shape)
print('테스트 target 레이블의 크기 :',decoder_target_test.shape)

훈련 source 데이터의 크기 : (29700, 8)
훈련 target 데이터의 크기 : (29700, 15)
훈련 target 레이블의 크기 : (29700, 15)
테스트 source 데이터의 크기 : (3300, 8)
테스트 target 데이터의 크기 : (3300, 15)
테스트 target 레이블의 크기 : (3300, 15)


## Model 설계

In [70]:
embedding_dim = 64
hidden_units = 64

In [71]:
encoder_inputs = Input(shape = (None,))
enc_emb = Embedding(src_vocab_size, embedding_dim)(encoder_inputs)

In [72]:
enc_emb

<KerasTensor: shape=(None, None, 64) dtype=float64 (created by layer 'embedding')>

In [73]:
# encoder를 bidirection LSTM으로 구현해 보았다. 
encoder_lstm = Bidirectional(LSTM(hidden_units, return_sequences=True, return_state = True))(enc_emb)

lstm, forward_h, forward_c, backward_h, backward_c = Bidirectional(LSTM(64, return_sequences=True, return_state = True))(encoder_lstm)

In [74]:
print(lstm.shape, forward_h.shape, forward_c.shape, backward_h.shape, backward_c.shape)

(None, None, 128) (None, 64) (None, 64) (None, 64) (None, 64)


In [75]:
state_h = Concatenate()([forward_h, backward_h]) # 은닉 상태
state_c = Concatenate()([forward_c, backward_c]) # 셀 상태

In [76]:
encoder_states = [state_h, state_c]

In [77]:
encoder_states

[<KerasTensor: shape=(None, 128) dtype=float64 (created by layer 'concatenate_6')>,
 <KerasTensor: shape=(None, 128) dtype=float64 (created by layer 'concatenate_7')>]

In [78]:
class BahdanauAttention(tf.keras.Model):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, values, query): # 단, key와 value는 같음
        # query shape == (batch_size, hidden size)
        # hidden_with_time_axis shape == (batch_size, 1, hidden size)
        # score 계산을 위해 뒤에서 할 덧셈을 위해서 차원을 변경해줍니다.
        hidden_with_time_axis = tf.expand_dims(query, 1)

        # score shape == (batch_size, max_length, 1)
        # we get 1 at the last axis because we are applying score to self.V
        # the shape of the tensor before applying self.V is (batch_size, max_length, units)
        score = self.V(tf.nn.tanh(
            self.W1(values) + self.W2(hidden_with_time_axis)))

        # attention_weights shape == (batch_size, max_length, 1)
        attention_weights = tf.nn.softmax(score, axis=1)

        # context_vector shape after sum == (batch_size, hidden_size)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

In [112]:
# encoder의 hidden state를 이용하여 context vector 계산
attention = BahdanauAttention(64) # 가중치 크기 정의
context_vector, attention_weights = attention(lstm, state_h)

In [113]:
context_vector

<KerasTensor: shape=(None, 128) dtype=float64 (created by layer 'bahdanau_attention_7')>

In [114]:
attention_weights

<KerasTensor: shape=(None, None, 1) dtype=float64 (created by layer 'bahdanau_attention_7')>

In [122]:
decoder_inputs = Input(shape=(1,))
dec_emb = Embedding(tar_vocab_size, embedding_dim)(decoder_inputs) # 임베딩 층

In [123]:
dec_emb

<KerasTensor: shape=(None, 1, 64) dtype=float64 (created by layer 'embedding_6')>

In [124]:
c = tf.concat([tf.expand_dims(context_vector, 1), dec_emb], axis = -1)
c

<KerasTensor: shape=(None, 1, 192) dtype=float64 (created by layer 'tf.concat_18')>

In [86]:
decoder_lstm = LSTM(128, return_sequences = True, return_state = True)

decoder_outputs, _, _ = decoder_lstm(c, initial_state = encoder_states)
decoder_outputs

<KerasTensor: shape=(None, 1, 128) dtype=float64 (created by layer 'lstm_2')>

In [87]:
decoder_dense = Dense(tar_vocab_size, activation = 'softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [88]:
model = Model(inputs = [encoder_inputs, decoder_inputs], outputs = decoder_outputs)

In [89]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['acc'])

In [90]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, None)]       0                                            
__________________________________________________________________________________________________
embedding (Embedding)           (None, None, 64)     299008      input_1[0][0]                    
__________________________________________________________________________________________________
bidirectional_5 (Bidirectional) [(None, None, 128),  66048       embedding[0][0]                  
__________________________________________________________________________________________________
bidirectional_6 (Bidirectional) [(None, None, 128),  98816       bidirectional_5[0][0]            
                                                                 bidirectional_5[0][1]        

In [91]:
model.fit(x=[encoder_input_train, decoder_input_train], y=decoder_target_train, \
          validation_data=([encoder_input_test, decoder_input_test], decoder_target_test),
          batch_size=128, epochs=30)

Epoch 1/30


ValueError: in user code:

    C:\Users\PC\anaconda3\lib\site-packages\tensorflow\python\keras\engine\training.py:805 train_function  *
        return step_function(self, iterator)
    C:\Users\PC\anaconda3\lib\site-packages\tensorflow\python\keras\engine\training.py:795 step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    C:\Users\PC\anaconda3\lib\site-packages\tensorflow\python\distribute\distribute_lib.py:1259 run
        return self._extended.call_for_each_replica(fn, args=args, kwargs=kwargs)
    C:\Users\PC\anaconda3\lib\site-packages\tensorflow\python\distribute\distribute_lib.py:2730 call_for_each_replica
        return self._call_for_each_replica(fn, args, kwargs)
    C:\Users\PC\anaconda3\lib\site-packages\tensorflow\python\distribute\distribute_lib.py:3417 _call_for_each_replica
        return fn(*args, **kwargs)
    C:\Users\PC\anaconda3\lib\site-packages\tensorflow\python\keras\engine\training.py:788 run_step  **
        outputs = m

In [106]:
encoder_input_train[0]

array([78,  6,  1,  0,  0,  0,  0,  0])

In [105]:
decoder_input_train[0]

array([551,  13,   7,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0])

**문제점** <br>
* input 데이터 자체가 원-핫 인코딩된 데이터가 들어오고 embedding을 하지 않는 형식으로 모델 설계를 진행해야 한다.
* embedding을 하고 싶으면 embedding을 받을 수 있는 형태로 input shape의 조정이 필요하다.
* decoder는 초반에 encoder의 state와 decoder의 초기 state(encoder의 hidden state로부터 생성되는)를 input으로 받는다.
* 이후에는 예측한 값이 input으로 들어와야하는 구조
* attention을 한 context vector도 decoder가 수행할 때마다 새로이 갱신되어야한다.

# 오픈소스
* https://www.kaggle.com/code/kmkarakaya/encoder-decoder-with-bahdanau-luong-attention

In [1]:
import tensorflow as tf

print('tf version : ', tf.__version__)
print('tf.keras version : ', tf.keras.__version__)

tf version :  2.4.1
tf.keras version :  2.4.0


In [2]:
from random import randint # random한 int값을 생성해주는 모듈

import numpy as np
from numpy import array, argmax, array_equal

from tensorflow.keras import models
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Bidirectional
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras import Input

from tensorflow.keras.layers import TimeDistributed
from tensorflow.keras.layers import RepeatVector
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import load_model

import matplotlib.pyplot as plt
import matplotlib.pyplot as ticker

from tensorflow.keras.layers import Lambda
from tensorflow.keras import backend as K

In [3]:
tf.keras.backend.set_floatx('float64')

In [4]:
def generate_sequence(length, n_unique):
    # length 만큼 randint 생성
    # n_unique : 사전에 있는 단어 수
    return [randint(1, n_unique - 1) for _ in range(length)]

def one_hot_encode(sequence, n_unique):
    encoding = list()
    
    # sequence에 숫자 리스트가 들어옴
    # 그 리스트의 value 숫자를 인덱스로 하여 해당 인덱스에 1
    for value in sequence:
        vector = [0 for _ in range(n_unique)] # 0 vector 생성
        vector[value] = 1
        encoding.append(vector)
        # encoding은 각 value의 원핫인코딩 결과를 담고 있음
        
    return array(encoding)

def one_hot_decode(encoded_seq):
    # one hot encoding된 sequence에서 최대값 가지는 index 반환 = 숫자
    return [argmax(vector) for vector in encoded_seq]

def get_reversed_pairs(time_steps, vocabulary_size, verbose = False):
    sequence_in = generate_sequence(time_steps, vocabulary_size)
    sequence_out = sequence_in[::-1]
    
    X = one_hot_encode(sequence_in, vocabulary_size)
    y = one_hot_encode(sequence_out, vocabulary_size)
    
    # 3D로 변경
    X = X.reshape((1, X.shape[0], X.shape[1]))
    y = y.reshape((1, y.shape[0], y.shape[1]))
    
    if (verbose):
        print('\nFor each input sequence (X), selecting ',time_steps,' random numbers beteen 1 and ',
              vocabulary_size, ' (0 is reserved )')
      
        print('\nA sample X ')
        print('X=%s' % (one_hot_decode(X[0])))
        print('\nreversed input sequence (X) is the output sequence (y) ')
        print('y=%s' % (one_hot_decode(y[0])))

        print('\nEach input and output sequences are converted one_hot_encoded format in ',
              vocabulary_size,' dimensions')
        print('X=%s' % (X[0]))
        print('y=%s' % (y[0]))
    
    return X, y

def create_dataset(train_size, test_size, time_steps, vocabulary_size, verbose = False):
    pairs = [get_reversed_pairs(time_steps, vocabulary_size) for _ in range(train_size)]
    pairs = np.array(pairs).squeeze()
    
    X_train = pairs[:, 0]
    y_train = pairs[:, 1]
    
    pairs = [get_reversed_pairs(time_steps, vocabulary_size) for _ in range(test_size)]
    pairs = np.array(pairs).squeeze()
    
    X_test = pairs[:, 0]
    y_test = pairs[:, 1]
    
    if (verbose):
        print('\nGenerated sequence datasets as follows (batch_size,time_steps, features)')
        print('X_train.shape: ', X_train.shape,'y_train.shape: ', y_train.shape)
        print('X_test.shape: ', X_test.shape,'y_test.shape: ', y_test.shape)
        
    return X_train, y_train, X_test, y_test

In [5]:
n_timesteps_in = 4
n_features = 10

sample_X, sample_y = get_reversed_pairs(n_timesteps_in, n_features, verbose = True)

train_size = 2000
test_size = 200

X_train, y_train, X_test, y_test = create_dataset(train_size, test_size, n_timesteps_in, n_features, verbose = True)


For each input sequence (X), selecting  4  random numbers beteen 1 and  10  (0 is reserved )

A sample X 
X=[5, 8, 8, 2]

reversed input sequence (X) is the output sequence (y) 
y=[2, 8, 8, 5]

Each input and output sequences are converted one_hot_encoded format in  10  dimensions
X=[[0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0]
 [0 0 1 0 0 0 0 0 0 0]]
y=[[0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 1 0 0 0 0]]

Generated sequence datasets as follows (batch_size,time_steps, features)
X_train.shape:  (2000, 4, 10) y_train.shape:  (2000, 4, 10)
X_test.shape:  (200, 4, 10) y_test.shape:  (200, 4, 10)


In [135]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units, verbose = 0):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)
        self.verbose = verbose
        
    def call(self, query, values):
        # qurey : decoder의 time -1의 hidden state 
        # values : encoder의 hidden state
        
        if self.verbose:
            print('\n******* Bahdanau Attention STARTS******')
            print('query (decoder hidden state): (batch_size, hidden size) ', query.shape)
            print('values (encoder all hidden state): (batch_size, max_len, hidden size) ', values.shape)
            
        query_with_time_axis = tf.expand_dims(query, 1)
        # score 계산 시 차원을 맞춰주기 위해서 expand_dims 사용
        
        if self.verbose:    
            print('query_with_time_axis:(batch_size, 1, hidden size) ', query_with_time_axis.shape)
        
        # score 계산 : source 문장에서 어디에 집중해야하는 가에 대한 점수
        score = self.V(tf.nn.tanh(
                    self.W1(query_with_time_axis) + self.W2(values)))
        
        if self.verbose:
            print('score: (batch_size, max_length, 1) ',score.shape)
        
        # softmax : 가중치 계산
        attention_weights = tf.nn.softmax(score, axis = 1)
        
        if self.verbose:
            print('attention_weights: (batch_size, max_length, 1) ',attention_weights.shape)
        
        context_vector = attention_weights * values
        
        if self.verbose:
            print('context_vector before reduce_sum: (batch_size, max_length, hidden_size) ',context_vector.shape)
            
        context_vector = tf.reduce_sum(context_vector, axis = 1)
        
        if self.verbose:
            print('context_vector after reduce_sum: (batch_size, hidden_size) ',context_vector.shape)
            print('\n******* Bahdanau Attention ENDS******')
            
        return context_vector, attention_weights

In [136]:
latentSpaceDimension = 16

In [137]:
encoder_states

[<KerasTensor: shape=(None, 32) dtype=float64 (created by layer 'concatenate_10')>,
 <KerasTensor: shape=(None, 32) dtype=float64 (created by layer 'concatenate_11')>]

In [142]:
verbose = 1

batch_size = 1

if verbose:
    print('***** Model Hyper Parameters *******')
    print('latentSpaceDimension: ', latentSpaceDimension)
    print('batch_size: ', batch_size)
    print('sequence length: ', n_timesteps_in)
    print('n_features: ', n_features)

    print('\n***** TENSOR DIMENSIONS *******')
    
    
encoder_inputs = Input(shape = (n_timesteps_in, n_features), name = 'encoder_inputs')
encoder_lstm = LSTM(latentSpaceDimension, return_sequences = True, return_state = True, name = 'encoder_lstm')
encoder_outputs, encoder_state_h, encoder_state_c = encoder_lstm(encoder_inputs)

# encoder_lstm = Bidirectional(LSTM(latentSpaceDimension, return_sequences = True, return_state = True, name = 'encoder_lstm'))
# encoder_outputs, encoder_fwd_h, encoder_fwd_c, encoder_bck_h, encoder_bck_c = encoder_lstm(encoder_inputs)

# encoder_state_h = Concatenate()([encoder_fwd_h, encoder_bck_h])
# encoder_state_c = Concatenate()([encoder_fwd_c, encoder_bck_c])
encoder_states = [encoder_state_h, encoder_state_c]

if verbose:
    print ('Encoder output shape: (batch size, sequence length, latentSpaceDimension) {}'.format(encoder_outputs.shape))
    print ('Encoder Hidden state shape: (batch size, latentSpaceDimension) {}'.format(encoder_state_h.shape))
    print ('Encoder Cell state shape: (batch size, latentSpaceDimension) {}'.format(encoder_state_c.shape))

if verbose:
    print(encoder_states)

***** Model Hyper Parameters *******
latentSpaceDimension:  16
batch_size:  1
sequence length:  4
n_features:  10

***** TENSOR DIMENSIONS *******
Encoder output shape: (batch size, sequence length, latentSpaceDimension) (None, 4, 16)
Encoder Hidden state shape: (batch size, latentSpaceDimension) (None, 16)
Encoder Cell state shape: (batch size, latentSpaceDimension) (None, 16)
[<KerasTensor: shape=(None, 16) dtype=float64 (created by layer 'encoder_lstm')>, <KerasTensor: shape=(None, 16) dtype=float64 (created by layer 'encoder_lstm')>]


In [143]:
# 축소할 dimension 크기 만큼의 context vector 생성하는 attention
attention = BahdanauAttention(latentSpaceDimension, verbose = verbose)

In [144]:
# decoder의 input : 문장 길이에 context vector길이만큼의 input으로 들어옴
decoder_inputs = Input(shape = (1, (n_features + latentSpaceDimension)), name = 'decoder_inputs')
decoder_lstm = LSTM(latentSpaceDimension, return_state = True, name = 'decoder_lstm')
decoder_dense = Dense(n_features, activation = 'softmax', name = 'decoder_dense')

all_outputs = []

# decoder의 첫번째 input값 = <sos>의 역할
inputs = np.zeros((batch_size, 1, n_features))
inputs[:, 0, 0] = 1

# 초기 decoder의 output은 마지막 encoder의 hidden state : s0 설정
decoder_outputs = encoder_state_h
states = encoder_states # 초기 decoder의 input은 마지막 encoder state

if verbose:
    print('initial decoder inputs : ', inputs.shape)

initial decoder inputs :  (1, 1, 10)


In [145]:
# 문장길이 = time만큼 반복
for _ in range(n_timesteps_in):
    
    context_vector, attention_weights = attention(decoder_outputs, encoder_outputs)
    
    if verbose:
        print("Attention context_vector: (batch size, units) {}".format(context_vector.shape))
        print("Attention weights : (batch_size, sequence_length, 1) {}".format(attention_weights.shape))
        print('decoder_outputs: (batch_size,  latentSpaceDimension) ', decoder_outputs.shape )

    context_vector = tf.expand_dims(context_vector, 1)
    
    if verbose:
        print('Reshaped context_vector : ', context_vector.shape)
        
    inputs = tf.concat([context_vector, inputs], axis = -1)
    
    if verbose:
        print('After concat inputs : ,(batch_size, 1, n_features + hidden_size): ',inputs.shape)
        
    decoder_outputs, state_h, state_c = decoder_lstm(inputs, initial_state = states) 
    # 1번째 cycle : states = encoder_states
    outputs = decoder_dense(decoder_outputs)
    # outputs는 feature마다의 확률값
    
    outputs = tf.expand_dims(outputs, 1)
    all_outputs.append(outputs)
    
    # 현재 time의 output이 다음 time의 decoder의 input이 되어야함 - outputs와 state를 변경해줌
    inputs = outputs
    states = [state_h, state_c]            


******* Bahdanau Attention STARTS******
query (decoder hidden state): (batch_size, hidden size)  (None, 16)
values (encoder all hidden state): (batch_size, max_len, hidden size)  (None, 4, 16)
query_with_time_axis:(batch_size, 1, hidden size)  (None, 1, 16)
score: (batch_size, max_length, 1)  (None, 4, 1)
attention_weights: (batch_size, max_length, 1)  (None, 4, 1)
context_vector before reduce_sum: (batch_size, max_length, hidden_size)  (None, 4, 16)
context_vector after reduce_sum: (batch_size, hidden_size)  (None, 16)

******* Bahdanau Attention ENDS******
Attention context_vector: (batch size, units) (None, 16)
Attention weights : (batch_size, sequence_length, 1) (None, 4, 1)
decoder_outputs: (batch_size,  latentSpaceDimension)  (None, 16)
Reshaped context_vector :  (None, 1, 16)
After concat inputs : ,(batch_size, 1, n_features + hidden_size):  (1, 1, 26)

******* Bahdanau Attention STARTS******
query (decoder hidden state): (batch_size, hidden size)  (None, 16)
values (encoder al

In [68]:
# 모든 예측 결과를 합침
decoder_outputs = Lambda(lambda x : K.concatenate(x, axis = 1))(all_outputs)

In [69]:
model = Model(encoder_inputs, decoder_outputs, name = 'model_encoder_decoder')

In [14]:
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

In [15]:
model.summary()

Model: "model_encoder_decoder"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
encoder_inputs (InputLayer)     [(None, 4, 10)]      0                                            
__________________________________________________________________________________________________
encoder_lstm (LSTM)             [(None, 4, 16), (Non 1728        encoder_inputs[0][0]             
__________________________________________________________________________________________________
bahdanau_attention (BahdanauAtt ((None, 16), (None,  561         encoder_lstm[0][1]               
                                                                 encoder_lstm[0][0]               
                                                                 decoder_lstm[0][0]               
                                                                 encoder_lstm[

**보완점 및 의문점** <br>
* encoder가 biLSTM이 됐을 때 forward와 backward가 합쳐지면서 dimension의 숫자가 변화
* context vector 생성 시에도 숫자 변화가 생김
* 예측한 y값과 context vector가 합쳐지게 되는데 이게 input으로 들어갈 때 상관이 있는지